# Limpieza y transformación de datos — Cafe Sales (Dirty Data)

**Dataset:** [Cafe Sales – Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training) (Kaggle)

**Objetivo:** cargar e inspeccionar la base, identificar los principales problemas de calidad, aplicar técnicas de limpieza y transformación con Python (Pandas/NumPy), justificar cada decisión y exportar una versión limpia del dataset.

Este notebook sigue el flujo:
1. Carga e inspección inicial.
2. Diagnóstico de problemas de calidad.
3. Limpieza y transformación (con justificación de cada paso).
4. Verificación de duplicados y valores atípicos.
5. Exportación de la base limpia.
6. Tabla resumen de problemas y decisiones.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

RAW_PATH = 'data/dirty_cafe_sales.csv'
OUT_PATH = 'data/cafe_sales_clean.csv'


## 1. Carga e inspección de la base

In [2]:
df = pd.read_csv(RAW_PATH)
print('Dimensiones:', df.shape)
df.head()


Dimensiones: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


Al inspeccionar `df.info()` se observa que **todas las columnas se cargaron como texto (`object`)**, incluyendo las que deberían ser numéricas (`Quantity`, `Price Per Unit`, `Total Spent`) y la fecha (`Transaction Date`). Esto ya es un primer indicio de problema de calidad: tipos de datos incorrectos.

In [4]:
for col in df.columns:
    print(f'--- {col} ---')
    print(df[col].unique()[:12])
    print('valores únicos:', df[col].nunique(), '\n')


--- Transaction ID ---
<StringArray>
['TXN_1961373', 'TXN_4977031', 'TXN_4271903', 'TXN_7034554', 'TXN_3160411',
 'TXN_2602893', 'TXN_4433211', 'TXN_6699534', 'TXN_4717867', 'TXN_2064365',
 'TXN_2548360', 'TXN_3051279']
Length: 12, dtype: str
valores únicos: 10000 

--- Item ---
<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str
valores únicos: 10 

--- Quantity ---
<StringArray>
['2', '4', '5', '3', '1', 'ERROR', 'UNKNOWN', nan]
Length: 8, dtype: str
valores únicos: 7 

--- Price Per Unit ---
<StringArray>
['2.0', '3.0', '1.0', '5.0', '4.0', '1.5', nan, 'ERROR', 'UNKNOWN']
Length: 9, dtype: str
valores únicos: 8 

--- Total Spent ---
<StringArray>
[  '4.0',  '12.0', 'ERROR',  '10.0',  '20.0',   '9.0',  '16.0',  '15.0',
  '25.0',   '8.0',   '5.0',   '3.0']
Length: 12, dtype: str
valores únicos: 19 

--- Payment Method ---
<StringArray>
['Credit Card', 'Cash', 'UNKNOWN

**Diagnóstico inicial:** además de valores nulos (`NaN`) reales, el dataset contiene dos marcadores de dato inválido usados como texto: `'ERROR'` y `'UNKNOWN'`. Estos no son valores nulos de pandas, así que no se cuentan con `isna()` mientras no se normalicen.

## 2. Diagnóstico cuantitativo de problemas de calidad

In [5]:
invalid_tokens = ['ERROR', 'UNKNOWN']

diagnostico = pd.DataFrame({
    'NaN_real': df.isna().sum(),
    'marca_ERROR': (df == 'ERROR').sum(),
    'marca_UNKNOWN': (df == 'UNKNOWN').sum(),
})
diagnostico['total_dato_faltante'] = diagnostico.sum(axis=1)
diagnostico


,NaN_real,marca_ERROR,marca_UNKNOWN,total_dato_faltante
Transaction ID,0,0,0,0
Item,333,292,344,969
Quantity,138,170,171,479
Price Per Unit,179,190,164,533
Total Spent,173,164,165,502
Payment Method,2579,306,293,3178
Location,3265,358,338,3961
Transaction Date,159,142,159,460


In [6]:
print('Duplicados de fila completa:', df.duplicated().sum())
print('Duplicados de Transaction ID:', df.duplicated(subset=['Transaction ID']).sum())


Duplicados de fila completa: 0
Duplicados de Transaction ID: 0


No se encontraron filas duplicadas ni valores repetidos de `Transaction ID`; se documenta la verificación aunque no requirió corrección.

**Resumen de problemas identificados:**
- **Valores faltantes** reales (`NaN`) en todas las columnas excepto `Transaction ID`.
- **Valores marcados como inválidos** (`'ERROR'`, `'UNKNOWN'`) usados como texto en lugar de nulos.
- **Tipos de datos incorrectos**: columnas numéricas y de fecha almacenadas como texto.
- **Posibles valores atípicos** en `Quantity`, `Price Per Unit` y `Total Spent` (se validan más adelante).
- **Duplicados**: verificados, no encontrados.


## 3. Limpieza y transformación

### 3.1 Normalizar los marcadores `'ERROR'` / `'UNKNOWN'` a `NaN`

Ambos textos representan la misma situación: un dato que no se pudo capturar o validar correctamente. Tratarlos como cadenas de texto distintas de `NaN` impediría detectarlos con `isna()`, distorsionaría los conteos de valores únicos y rompería la conversión numérica. Se decide **unificarlos como `NaN`** antes de cualquier otro paso, para tener un único criterio de "dato faltante".

In [7]:
celdas_afectadas_tokens = int((df.isin(invalid_tokens)).sum().sum())
df = df.replace(invalid_tokens, np.nan)
print("Celdas con 'ERROR'/'UNKNOWN' convertidas a NaN:", celdas_afectadas_tokens)


Celdas con 'ERROR'/'UNKNOWN' convertidas a NaN: 3256


### 3.2 Corregir tipos de datos

`Quantity`, `Price Per Unit` y `Total Spent` deben ser numéricos; `Transaction Date` debe ser tipo fecha. Se usan `pd.to_numeric()` y `pd.to_datetime()` con `errors='coerce'`, que convierten a `NaN`/`NaT` cualquier valor no interpretable (en este caso ninguno adicional, porque ya se normalizaron los marcadores de texto en el paso anterior).

In [8]:
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

df[['Item', 'Payment Method', 'Location']] = df[['Item', 'Payment Method', 'Location']].astype('string')

df.dtypes


Transaction ID                 str
Item                        string
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              string
Location                    string
Transaction Date    datetime64[us]
dtype: object

### 3.3 Reconstrucción de valores faltantes usando la lógica del negocio

Antes de imputar con medias o modas genéricas, se aprovecha una regularidad real del dataset: **cada producto (`Item`) tiene un precio unitario fijo** en todo el conjunto, y se cumple siempre `Total Spent = Quantity × Price Per Unit`. Esto permite reconstruir valores faltantes de forma algebraica, en lugar de estimarlos estadísticamente, siempre que se conserven al menos dos de los tres datos relacionados (producto/cantidad/precio/total).

In [9]:
catalogo_precios = (
    df.dropna(subset=['Item', 'Price Per Unit'])
    .groupby('Item')['Price Per Unit']
    .agg(lambda s: s.mode().iloc[0])
    .to_dict()
)
catalogo_precios


{'Cake': 3.0,
 'Coffee': 2.0,
 'Cookie': 1.0,
 'Juice': 3.0,
 'Salad': 5.0,
 'Sandwich': 4.0,
 'Smoothie': 4.0,
 'Tea': 1.5}

In [10]:
# Verificación: ¿el precio identifica siempre al mismo producto?
precio_a_items = {}
for item, precio in catalogo_precios.items():
    precio_a_items.setdefault(precio, set()).add(item)
precio_a_items


{3.0: {'Cake', 'Juice'},
 2.0: {'Coffee'},
 1.0: {'Cookie'},
 5.0: {'Salad'},
 4.0: {'Sandwich', 'Smoothie'},
 1.5: {'Tea'}}

Se observa que los precios **1.0, 1.5, 2.0 y 5.0 identifican a un único producto** (Cookie, Tea, Coffee y Salad respectivamente), mientras que **3.0** (Cake/Juice) y **4.0** (Smoothie/Sandwich) son ambiguos. Por eso el `Item` solo se recupera a partir del precio cuando este es inequívoco; en los demás casos no se adivina.

In [11]:
mapa_precio_unico = {p: list(items)[0] for p, items in precio_a_items.items() if len(items) == 1}

# --- Recuperar Item a partir de un precio inequívoco ---
mask_item_na = df['Item'].isna() & df['Price Per Unit'].notna()
n_item_recuperado = 0
for precio, item in mapa_precio_unico.items():
    m = mask_item_na & np.isclose(df['Price Per Unit'], precio)
    n_item_recuperado += int(m.sum())
    df.loc[m, 'Item'] = item
print('Item recuperado a partir de un precio único:', n_item_recuperado)

# --- Recuperar Price Per Unit a partir del catálogo por Item ---
mask_price_na = df['Price Per Unit'].isna() & df['Item'].notna()
n_price_recuperado = int(mask_price_na.sum())
df.loc[mask_price_na, 'Price Per Unit'] = df.loc[mask_price_na, 'Item'].map(catalogo_precios)
print('Price Per Unit recuperado a partir del Item:', n_price_recuperado)


Item recuperado a partir de un precio único: 468
Price Per Unit recuperado a partir del Item: 479


In [12]:
q, pr, tt = df['Quantity'], df['Price Per Unit'], df['Total Spent']

# Total = Quantity * Price
mask_total = tt.isna() & q.notna() & pr.notna()
n_total = int(mask_total.sum())
df.loc[mask_total, 'Total Spent'] = q[mask_total] * pr[mask_total]

# Quantity = Total / Price
mask_qty = q.isna() & tt.notna() & pr.notna() & (pr != 0)
n_qty = int(mask_qty.sum())
df.loc[mask_qty, 'Quantity'] = (tt[mask_qty] / pr[mask_qty]).round()

# Price = Total / Quantity
mask_price = pr.isna() & tt.notna() & q.notna() & (q != 0)
n_price = int(mask_price.sum())
df.loc[mask_price, 'Price Per Unit'] = tt[mask_price] / q[mask_price]

print('Total Spent reconstruido (Quantity x Price):', n_total)
print('Quantity reconstruido (Total / Price):', n_qty)
print('Price Per Unit reconstruido (Total / Quantity):', n_price)


Total Spent reconstruido (Quantity x Price): 479
Quantity reconstruido (Total / Price): 456
Price Per Unit reconstruido (Total / Quantity): 48


In [13]:
# Segunda pasada: tras reconstruir precios, se puede recuperar algún Item adicional
mask_item_na2 = df['Item'].isna() & df['Price Per Unit'].notna()
n_item_recuperado2 = 0
for precio, item in mapa_precio_unico.items():
    m = mask_item_na2 & np.isclose(df['Price Per Unit'], precio)
    n_item_recuperado2 += int(m.sum())
    df.loc[m, 'Item'] = item
print('Item recuperado en la 2a pasada:', n_item_recuperado2)


Item recuperado en la 2a pasada: 21


### 3.4 Filas sin ningún dato de venta recuperable

Se define un criterio explícito para eliminar filas: solo se descartan aquellas en las que **`Item`, `Quantity`, `Price Per Unit` y `Total Spent` están vacíos simultáneamente**, es decir, filas que no aportan ninguna información de venta aprovechable. Cualquier fila con al menos un dato de venta se conserva.

In [14]:
core_cols = ['Item', 'Quantity', 'Price Per Unit', 'Total Spent']
mask_irrecuperable = df[core_cols].isna().all(axis=1)
n_irrecuperable = int(mask_irrecuperable.sum())
df = df[~mask_irrecuperable].copy()
print('Filas eliminadas por no tener ningún dato de venta recuperable:', n_irrecuperable)


Filas eliminadas por no tener ningún dato de venta recuperable: 0


### 3.5 Valores residuales que **no** se imputan

Después de la reconstrucción algebraica, quedan muy pocos valores faltantes en `Item`, `Quantity`, `Price Per Unit` y `Total Spent` para los que no existe ninguna relación matemática ni de catálogo que permita recuperarlos sin adivinar. En vez de rellenarlos con la media/moda global (lo que inventaría ventas que no ocurrieron), se **dejan como nulos** y se marcan con una columna booleana `Venta_incompleta` para que cualquier análisis posterior decida si los incluye o los excluye.

In [15]:
df['Venta_incompleta'] = df[core_cols].isna().any(axis=1)

print('Item nulo restante:', df['Item'].isna().sum())
print('Quantity nulo restante:', df['Quantity'].isna().sum())
print('Price Per Unit nulo restante:', df['Price Per Unit'].isna().sum())
print('Total Spent nulo restante:', df['Total Spent'].isna().sum())
print('Filas marcadas como Venta_incompleta:', df['Venta_incompleta'].sum())


Item nulo restante: 480
Quantity nulo restante: 23
Price Per Unit nulo restante: 6
Total Spent nulo restante: 23
Filas marcadas como Venta_incompleta: 500


### 3.6 `Payment Method` y `Location`: demasiados nulos para imputar con confianza

`Payment Method` y `Location` tienen un porcentaje muy alto de valores faltantes (~32 % y ~40 % respectivamente). Imputarlos con la moda sesgaría fuertemente cualquier análisis de métodos de pago o de canal de venta, y eliminar esas filas descartaría información de venta (producto, cantidad, fecha) que sí es válida. Se opta por una **categoría explícita `"Desconocido"`**, que preserva la fila y es honesta sobre la falta de dato (en vez de mentir con un valor inventado).

In [16]:
for col in ['Item', 'Payment Method', 'Location']:
    n_na = int(df[col].isna().sum())
    if n_na:
        df[col] = df[col].fillna('Desconocido')
    print(f'{col}: {n_na} nulos residuales -> "Desconocido"')


Item: 480 nulos residuales -> "Desconocido"
Payment Method: 3178 nulos residuales -> "Desconocido"
Location: 3961 nulos residuales -> "Desconocido"


### 3.7 Fechas faltantes

No existe ninguna otra columna que permita inferir la fecha real de una transacción, así que las fechas faltantes **se dejan como `NaT`** en vez de imputarse. Inventar una fecha alteraría cualquier análisis de estacionalidad o de tendencia temporal.

In [17]:
print('Transaction Date con NaT (no recuperable):', df['Transaction Date'].isna().sum())


Transaction Date con NaT (no recuperable): 460


## 4. Verificación final de duplicados y valores atípicos

In [18]:
print('Duplicados de fila completa (post-limpieza):', df.duplicated().sum())
print('Duplicados de Transaction ID (post-limpieza):', df.duplicated(subset=['Transaction ID']).sum())


Duplicados de fila completa (post-limpieza): 0
Duplicados de Transaction ID (post-limpieza): 0


In [19]:
df[['Quantity', 'Price Per Unit', 'Total Spent']].describe()


,Quantity,Price Per Unit,Total Spent
count,9977.000000,9994.000000,9977.000000
mean,3.024957,2.947018,8.930139
std,1.420395,1.280006,6.004921
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000


In [20]:
anios_fuera_2023 = int(((df['Transaction Date'].dt.year != 2023) & df['Transaction Date'].notna()).sum())
print('Fechas fuera del rango 2023:', anios_fuera_2023)


Fechas fuera del rango 2023: 0


**Conclusión sobre valores atípicos:** `Quantity` va de 1 a 5, `Price Per Unit` coincide siempre con el catálogo de precios (1.0 a 5.0) y `Total Spent` (1 a 25) es consistente con `Quantity x Price`. Todas las fechas válidas caen dentro de 2023. No se encontraron valores fuera de los rangos de negocio esperados, por lo que **no se aplicó ningún filtro de outliers (IQR u otro)**: hacerlo sin evidencia real de anomalía habría eliminado datos válidos sin justificación.

## 5. Exportar la base limpia

In [21]:
df.to_csv(OUT_PATH, index=False)
print('Filas finales:', len(df))
print('Guardado en:', OUT_PATH)
df.head()


Filas finales: 10000
Guardado en: data/cafe_sales_clean.csv


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Venta_incompleta
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,False
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,False
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,False
3,TXN_7034554,Salad,2.0,5.0,10.0,Desconocido,Desconocido,2023-04-27,False
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,False


## 6. Tabla resumen de problemas y decisiones

La siguiente tabla resume, para cada problema de calidad identificado, cuántos registros/celdas afectó, qué acción se tomó y por qué. Se genera también como archivo `tabla_resumen_limpieza.csv` en el repositorio.

In [22]:
resumen = pd.DataFrame([
    {
        'Problema encontrado': "Valores marcados como 'ERROR'/'UNKNOWN' (no son NaN de pandas)",
        'Registros afectados': celdas_afectadas_tokens,
        'Acción realizada': "Reemplazo global por NaN (df.replace)",
        'Justificación': "Unificar el criterio de dato faltante antes de imputar; dejarlos como texto rompería isna() y la conversión numérica.",
    },
    {
        'Problema encontrado': "Tipos de datos incorrectos (numéricos y fecha guardados como texto)",
        'Registros afectados': "10,000 filas x 4 columnas (Quantity, Price Per Unit, Total Spent, Transaction Date)",
        'Acción realizada': "Conversión con pd.to_numeric()/pd.to_datetime()",
        'Justificación': "Permitir operaciones aritméticas y de fecha, y detectar automáticamente valores no interpretables.",
    },
    {
        'Problema encontrado': "Valores faltantes en Item",
        'Registros afectados': 969,
        'Acción realizada': f"{n_item_recuperado + n_item_recuperado2} recuperados cruzando el precio único del catálogo; el resto marcado como 'Desconocido'",
        'Justificación': "Cuando el precio es ambiguo (3.0 = Cake o Juice) no hay evidencia suficiente para asignar un producto sin adivinar.",
    },
    {
        'Problema encontrado': "Valores faltantes en Price Per Unit",
        'Registros afectados': 533,
        'Acción realizada': f"{n_price_recuperado} recuperados desde el catálogo Item->Precio; {n_price} recuperados como Total/Quantity; 6 quedaron nulos",
        'Justificación': "El precio por producto es constante en todo el dataset: reconstruirlo desde el catálogo es una inferencia segura, no una invención.",
    },
    {
        'Problema encontrado': "Valores faltantes en Quantity",
        'Registros afectados': 479,
        'Acción realizada': f"{n_qty} reconstruidos como Total Spent / Price Per Unit; 23 quedaron nulos",
        'Justificación': "La relación Total = Cantidad x Precio se cumple en el 100% de los registros completos originales: es una reconstrucción algebraica confiable.",
    },
    {
        'Problema encontrado': "Valores faltantes en Total Spent",
        'Registros afectados': 502,
        'Acción realizada': f"{n_total} reconstruidos como Quantity x Price Per Unit; 23 quedaron nulos",
        'Justificación': "Misma relación algebraica verificada sobre los datos originales completos.",
    },
    {
        'Problema encontrado': "Duplicados (fila completa y Transaction ID)",
        'Registros afectados': 0,
        'Acción realizada': "Verificación con duplicated(); no se encontraron duplicados, no se eliminó ninguna fila por esta causa",
        'Justificación': "Cada Transaction ID es único por diseño; se documenta la verificación aunque no requirió corrección.",
    },
    {
        'Problema encontrado': "Valores faltantes en Payment Method",
        'Registros afectados': 3178,
        'Acción realizada': "Reemplazo por categoría explícita 'Desconocido'",
        'Justificación': "~32% de nulos; imputar con la moda sesgaría el análisis de métodos de pago, y eliminar filas perdería datos válidos de venta.",
    },
    {
        'Problema encontrado': "Valores faltantes en Location",
        'Registros afectados': 3961,
        'Acción realizada': "Reemplazo por categoría explícita 'Desconocido'",
        'Justificación': "~40% de nulos; mismo criterio que Payment Method: preservar la fila sin inventar el canal de venta.",
    },
    {
        'Problema encontrado': "Fechas faltantes en Transaction Date",
        'Registros afectados': 460,
        'Acción realizada': "Se dejaron como NaT (no se imputó ninguna fecha)",
        'Justificación': "No existe ninguna columna que permita inferir la fecha real; inventar una fecha alteraría análisis de estacionalidad.",
    },
    {
        'Problema encontrado': "Filas sin ningún dato de venta recuperable",
        'Registros afectados': n_irrecuperable,
        'Acción realizada': "Eliminación de filas (0 casos encontrados)",
        'Justificación': "Umbral definido para descartar filas verdaderamente inservibles; no fue necesario aplicarlo en este dataset.",
    },
    {
        'Problema encontrado': "Valores atípicos en Quantity / Price Per Unit / Total Spent / fechas",
        'Registros afectados': 0,
        'Acción realizada': "Revisión de rangos (describe(), catálogo de precios, año de las fechas); no se aplicó ningún filtro",
        'Justificación': "Todos los valores caen dentro de los rangos de negocio esperados; eliminar sin evidencia real de anomalía descartaría datos válidos.",
    },
])

resumen.to_csv('tabla_resumen_limpieza.csv', index=False)
resumen


,Problema encontrado,Registros afectados,Acción realizada,Justificación
0,Valores marcados como 'ERROR'/'UNKNOWN' (no so...,3256,Reemplazo global por NaN (df.replace),Unificar el criterio de dato faltante antes de...
1,Tipos de datos incorrectos (numéricos y fecha ...,"10,000 filas x 4 columnas (Quantity, Price Per...",Conversión con pd.to_numeric()/pd.to_datetime(),"Permitir operaciones aritméticas y de fecha, y..."
2,Valores faltantes en Item,969,489 recuperados cruzando el precio único del c...,Cuando el precio es ambiguo (3.0 = Cake o Juic...
3,Valores faltantes en Price Per Unit,533,479 recuperados desde el catálogo Item->Precio...,El precio por producto es constante en todo el...
4,Valores faltantes en Quantity,479,456 reconstruidos como Total Spent / Price Per...,La relación Total = Cantidad x Precio se cumpl...
5,Valores faltantes en Total Spent,502,479 reconstruidos como Quantity x Price Per Un...,Misma relación algebraica verificada sobre los...
6,Duplicados (fila completa y Transaction ID),0,Verificación con duplicated(); no se encontrar...,Cada Transaction ID es único por diseño; se do...
7,Valores faltantes en Payment Method,3178,Reemplazo por categoría explícita 'Desconocido',~32% de nulos; imputar con la moda sesgaría el...
8,Valores faltantes en Location,3961,Reemplazo por categoría explícita 'Desconocido',~40% de nulos; mismo criterio que Payment Meth...
9,Fechas faltantes en Transaction Date,460,Se dejaron como NaT (no se imputó ninguna fecha),No existe ninguna columna que permita inferir ...
